In [1]:
from steps.discover import DiscoverStep
from dotenv import load_dotenv

load_dotenv()

step = DiscoverStep(
    model_name="qwen3:8b",
    mcp_url="https://wd-mcp.wmcloud.org/mcp/",
)
state = await step.run(
    {"question": "Get all planets that have more than 2 moons."}
)


=== Step 1: Discover ===

=== Step 1.1: Search ===
  [tool-call] search_items args={'query': 'Get all planets that have more than 2 moons.'}
  [tool-call] search_properties args={'query': 'Get all planets that have more than 2 moons.'}
    [tool-call-stream] search_items args={"query": "planet", "lang": "en"}
    [model-usage] prompt_tokens=3421 output_tokens=2519 total_tokens=5940
    [tool-call] search_items args={'query': 'planet', 'lang': 'en'}
    [model-stream]
[{'type': 'text', 'text': 'Q2: Earth — third planet from the Sun in the Solar System\nQ634: planet — celestial body directly orbiting a star or stellar remnant\nQ7201060: Planet — Welsh cultural and political magazine\nQ324: Uranus — seventh planet in the Solar System, mainly composed of hydrogen and helium\nQ128207: terrestrial planet — planet that is composed primarily of silicate rocks or metals\nQ111: Mars — fourth planet from the Solar System, tellurian and orange-red due to iron oxide\nQ11972514: Green Planet — Norw

In [ ]:
print(state['discovery_summary'])

In [ ]:
from steps.validate_discover import ValidateDiscoveryStep

step = ValidateDiscoveryStep(
    model_name="qwen3:8b",
    mcp_url="https://wd-mcp.wmcloud.org/mcp/",
)
state2 = await step.run(state)

In [ ]:
from steps.generate_sparql import GenerateSparqlStep

step = GenerateSparqlStep(
    model_name="qwen3:8b",
    mcp_url="https://wd-mcp.wmcloud.org/mcp/",
)

state3 = await step.run(state2)

In [ ]:
print(state3['sparql'])

In [ ]:
print(state3['sparql_results'])

In [ ]:
import requests
import os

WD_QUERY_URI = os.environ.get("WD_QUERY_URI", "https://query.wikidata.org/sparql")
USER_AGENT = os.environ.get("USER_AGENT", "Wikidata MCP SPARQL Generation (embedding@wikimedia.de)")

def execute_sparql(sparql: str) -> dict:
    """Execute a SPARQL query against the Wikidata endpoint."""
    result = requests.get(
        WD_QUERY_URI,
        params={
            "query": sparql,
            "format": "json",
        },
        headers={"User-Agent": f"{USER_AGENT}"}
    )

    if result.status_code == 400:
        error_message = result.text.split("	at ")[0]
        raise ValueError(error_message)
    result.raise_for_status()

    result_bindings = result.json()["results"]["bindings"]
    return result_bindings

execute_sparql(state3['sparql'])

In [ ]:
import requests
import json
import os

response = requests.post(
  url="https://openrouter.ai/api/v1/chat/completions",
  headers={
    "Authorization": "Bearer " + os.getenv("OPENROUTER_API_KEY"),
    "Content-Type": "application/json",
  },
  data=json.dumps({
    "model": "google/gemma-4-31b-it:free",
    "messages": [
      {
        "role": "user",
        "content": "What is the meaning of life?"
      }
    ]
  })
)